<a href="https://colab.research.google.com/github/AndreiDragomir07/COS484_A4/blob/main/Andrei_Dragomir_A4_Programming_Cross_Entropy_Loss_on_GPU_(COS484_S2026).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook for Programming in Problem 4
Welcome to the programming portion of the assignment! Each assignment throughout the semester will have a written portion and a programming portion. We will be using [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb#recent=true), so if you have never used it before, take a quick look through this introduction: [Working with Google Colab](https://docs.google.com/document/d/1LlnXoOblXwW3YX-0yG_5seTXJsb3kRdMMRYqs8Qqum4/edit?usp=sharing).

## Cross Entropy Loss on GPU
In this problem you will benchmark the cross-entropy loss in PyTorch and measure its efficiency relative to the theoretical memory bandwidth of the GPU. You will predict performance before measuring, then explain any discrepancies — this is the core skill of systems performance analysis.
Setup: Use the following parameters throughout:

-  Batch size: B = 4,096
-  Vocabulary size: V = 128,256
-  Logits dtype: torch.bfloat16
-  Reduction: ’none’
- Device: GPU (use a Colab GPU runtime, e.g., T4 or A100)
-  Report your PyTorch version (e.g., torch. version ) and GPU model in your notebook. Results can vary significantly across versions.

## Generate random input logits and targets

In [1]:
# Generate random input logits and targets:
import torch
B, V = 4096, 128256
logits = torch.randn(B, V, dtype=torch.bfloat16 , device='cuda',
requires_grad=True)
targets = torch.randint(0, V, (B,), device='cuda')

# Questions

**(a) Predict before you measure. (8 points)**

Before writing any benchmarking code, use your answers from Theory Problem 1 to predict the performance of an ideal (perfectly memory-bound) cross-entropy kernel on your GPU.

(i) Look up the peak memory bandwidth of your Colab GPU. For this assignment, if you are using an A100, assume the A100 SXM bandwidth of ∼2039 GB/s; if you are using a T4, use ∼300 GB/s. Using the total bytes from Theory Problem 1(c), compute the minimum possible time (in ms) for the forward pass and backward pass, assuming the kernel runs at 100% of peak memory bandwidth. (4 points)

(ii) Based on your arithmetic intensity analysis from Theory Problem 1(c), explain why memory bandwidth (not TFLOPS) is the right performance limiter and the right metric to evaluate the efficiency of cross- entropy. (4 points)

**Answers**

(i) The total number of bytes loaded from and stored to the memory of the GPU during the forward pass is 1.05 GB. I will be using a A100 runtime, so the time computation would take for a perfectly memory bound kernel is:

$ \frac{1.05 GB}{2039 GB/s} \approx 0.515 ms$

For the backward pass, the total bytes are 2.10 GB, so the ideal time is:

$ \frac{2.10 GB}{2039 GB/s} \approx 1.030 ms$

Thus, in total, it would take $1.545 ms$ to perform both passes.

(ii) The arithmetic intensity of the cross-entropy computation is approximately two orders of magnitude lower than the compute-to-bandwidth ratio of the kernel, which means that also the time it takes the kernel to perform all the compute is approximately two orders of magnitude lower than the time it takes it to move data to and from memory.

Thus, memory bandwidth is the practical performance limiter of the computation.


**(b) Benchmark PyTorch eager mode. (12 points)**


(i) Write a benchmarking function that measures the wall-clock time (in milliseconds) for the cross-entropy forward pass and backward pass separately. Use torch.cuda.Event with enable timing=True for accurate GPU timing. After recording the end event, make sure to synchronize the GPU (e.g., with torch.cuda.synchronize() or end event.synchronize()) before reading the elapsed time. Run each measurement multiple times (e.g., 100 iterations) and report the median time. Make sure to do a few warmup iterations before timing. (5 points)


(ii) Compute the achieved memory bandwidth (in GB/s) for the forward and backward passes: Achieved BW (GB/s) = Total bytes (GB) / Time (s)
    
How does the measured time compare to your prediction from part (a)? If there is a significant gap, explain why PyTorch eager mode is slower than the ideal. What extra memory traffic or overhead might the eager implementation incur? (7 points)

Hint: Think about whether the eager implementation fuses all the operations into a single kernel, or whether it materializes intermediate tensors (e.g., the full softmax output) to HBM.

**Answers**

(i) Median time for forward pass: 3.04 ms

Median time for backward pass: 6.08 ms

(ii) Forward pass achieved memory bandwidth: $ \frac{1.05GB}{3.04ms} \approx 345 GB/s $

Backward pass achieved memory bandwidth:
$ \frac{2.10GB}{6.08ms} \approx 345 GB/s $

<br>

The measured time is around 6 times higher than the predicted one from part (a). Most probably the significant gap between the predicted and achieved running times is caused by the fact the Python's eager implementation stores all intermediary tensors obtained in computations (such as the softmax output) to HBM, something that the fused kernel we assumed for the theory part of the assignment does not. Therefore, the traffic of data in this case is not as efficiently handled as it could be.




Code for part (b):

In [2]:
import torch
from torch import nn
from statistics import median

loss = nn.CrossEntropyLoss(reduction='none')

start_timer = torch.cuda.Event(enable_timing=True)
end_timer = torch.cuda.Event(enable_timing=True)

num_warmup = 5
num_iterations = 100

# --- Forward Pass Benchmarking ---
forward_times = []
print(f"Benchmarking forward pass with {num_warmup} warmup and {num_iterations} iterations...")
for i in range(num_warmup + num_iterations):
    if logits.grad is not None:
        logits.grad.zero_()
    start_timer.record()
    output = loss(logits, targets)
    end_timer.record()
    torch.cuda.synchronize()
    if i >= num_warmup:
        forward_times.append(start_timer.elapsed_time(end_timer))

median_forward_time = median(forward_times)
print(f"Median time for forward pass: {median_forward_time:.2f} ms")

# --- Backward Pass Benchmarking ---
backward_times = []
print(f"\nBenchmarking backward pass with {num_warmup} warmup and {num_iterations} iterations...")
for i in range(num_warmup + num_iterations):
    # Re-run forward pass to create a fresh computational graph for each backward pass
    if logits.grad is not None:
        logits.grad.zero_()
    output_for_backward = loss(logits, targets)

    start_timer.record()
    output_for_backward.sum().backward()
    end_timer.record()
    torch.cuda.synchronize()
    if i >= num_warmup:
        backward_times.append(start_timer.elapsed_time(end_timer))

median_backward_time = median(backward_times)
print(f"Median time for backward pass: {median_backward_time:.2f} ms")

Benchmarking forward pass with 5 warmup and 100 iterations...
Median time for forward pass: 3.08 ms

Benchmarking backward pass with 5 warmup and 100 iterations...
Median time for backward pass: 6.22 ms


**(c) Benchmark torch.compile mode. (10 points)**

(i) Wrap the cross-entropy computation with torch.compile and repeat the benchmarking from part (b). Note that the first call triggers compilation — make sure your warmup accounts for this. Report the achieved memory bandwidth. (4 points)


(ii) How does torch.compile compare to eager mode and to your ideal prediction? Explain what torch.compile is likely doing differently (e.g., kernel fusion) that accounts for the performance difference. (6 points)

**Answers**

(i) Forward pass running time: 1.56 ms

Backward pass running time: 4.01 ms

Therefore, the achieved bandwidths are:

Forward - $\approx 673 GB/s$

Backward - $\approx 524 GB/s $

(ii) torch.compile is significantly better than the eager mode - somewhere between 1.5 to 2 times faster bandwidth. However, it is still way below the ideal prediction.

torch.compile likely fuses the kernel at some points in the computational graph of the forward and backward passes, and thus avoids writing intermediary values to the HBM. One can notice that the running time of the forward pass isn't approximately half of the running time of the backward pass anymore, which indicates that the torch.compile wrapper doesn't optimize the two passes symmetrically. Therefore, the optimization probably depends on the computational graph.



Code for part (c):

In [3]:
import torch
from torch import nn
from statistics import median

# Define the cross-entropy loss function
def ce_loss_fn(logits, targets):
    return nn.CrossEntropyLoss(reduction='none')(logits, targets)

# Wrap the function with torch.compile
compiled_ce_loss_fn = torch.compile(ce_loss_fn)

start_timer = torch.cuda.Event(enable_timing=True)
end_timer = torch.cuda.Event(enable_timing=True)

num_warmup = 5
num_iterations = 100

# --- Compiled Forward Pass Benchmarking ---
compiled_forward_times = []
print(f"Benchmarking COMPILED forward pass with {num_warmup} warmup and {num_iterations} iterations...")
for i in range(num_warmup + num_iterations):
    if logits.grad is not None:
        logits.grad.zero_()
    start_timer.record()
    output = compiled_ce_loss_fn(logits, targets)
    end_timer.record()
    torch.cuda.synchronize()
    if i >= num_warmup:
        compiled_forward_times.append(start_timer.elapsed_time(end_timer))

median_compiled_forward_time = median(compiled_forward_times)
print(f"Median time for COMPILED forward pass: {median_compiled_forward_time:.2f} ms")

# --- Compiled Backward Pass Benchmarking ---
compiled_backward_times = []
print(f"\nBenchmarking COMPILED backward pass with {num_warmup} warmup and {num_iterations} iterations...")
for i in range(num_warmup + num_iterations):
    # Re-run forward pass to create a fresh computational graph for each backward pass
    if logits.grad is not None:
        logits.grad.zero_()
    output_for_backward = compiled_ce_loss_fn(logits, targets)

    start_timer.record()
    output_for_backward.sum().backward()
    end_timer.record()
    torch.cuda.synchronize()
    if i >= num_warmup:
        compiled_backward_times.append(start_timer.elapsed_time(end_timer))

median_compiled_backward_time = median(compiled_backward_times)
print(f"Median time for COMPILED backward pass: {median_compiled_backward_time:.2f} ms")

Benchmarking COMPILED forward pass with 5 warmup and 100 iterations...
Median time for COMPILED forward pass: 1.59 ms

Benchmarking COMPILED backward pass with 5 warmup and 100 iterations...
Median time for COMPILED backward pass: 4.00 ms



**(d) Inspecting the compiled kernel. (10 points)**

When torch.compile compiles your function, it generates Triton kernel code under the hood.

(i) Use the environment variable TORCH LOGS=output code or torch. dynamo.config.log level to dump the generated Triton kernel code for the cross-entropy forward pass. Include the generated code (or relevant excerpts) in your notebook. (3 points)


(ii) Annotate the generated kernel: identify which lines correspond to (1) computing the max for numer- ical stability, (2) the exp and sum (softmax denominator), (3) the log-sum-exp, and (4) the final loss computation. Relate these back to the formulas you derived in Theory Problem 1(a). (7 points)

TODO: ANSWER THE QUESTION HERE (DOUBLE-CLICK TO EDIT)



Code for part (d):

In [5]:
import torch
import torch.nn as nn
import logging
import torch.dynamo # Re-adding explicit import for clarity and safety

# Set the log level to debug for torch.dynamo to dump the generated code
torch.dynamo.config.log_level = logging.DEBUG # Corrected from torch._dynamo.config
torch.dynamo.config.output_code = True # Corrected from torch._dynamo.config

# Define the cross-entropy loss function (as in part c)
def ce_loss_fn(logits, targets):
    return nn.CrossEntropyLoss(reduction='none')(logits, targets)

# Wrap the function with torch.compile (as in part c)
compiled_ce_loss_fn = torch.compile(ce_loss_fn)

# Retrieve logits and targets from the previous cell's execution context
# (Assuming they are still in scope from cell lU5OBVr5zG34)

# Run the compiled function once to trigger compilation and log output
print("Running compiled forward pass to dump Triton kernel code...")
output_dump = compiled_ce_loss_fn(logits, targets)

print("Triton kernel code has been dumped above. Scroll up to see the output.")

ModuleNotFoundError: No module named 'torch.dynamo'

In [20]:
# Uninstall existing PyTorch packages
!pip uninstall torch torchvision torchaudio -y

# Install PyTorch with the correct CUDA version (cu121 as per Colab's typical setup)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

print("PyTorch reinstallation complete. Please restart the runtime.")

Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (780.4 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-linux_x86_64.whl (7.3 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchaudio-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (3.4 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl (664.8 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cublas_cu12-1

PyTorch reinstallation complete. Please restart the runtime.


**IMPORTANT:** After the above cell finishes, please go to the Colab menu: `Runtime > Restart runtime`. Once the runtime has restarted, you can verify `torch.dynamo` availability by running the check cell again, and then proceed with part (d) of your problem.

# Problem 2: Bonus — Cross-Entropy Speed Competition

See task specification and submission instruction in the problem set PDF. Submission window is open from April 7 to April 16. If you're submitting for bonus credits, please include your optimization journal here documenting your optimization process:


- What approaches did you try? (e.g., torch.compile tricks, custom Triton kernels, fused forward+backward,
memory layout changes, block size tuning, etc.)
- For each approach, report the timing and explain why it was faster or slower than your previous best.
- What is the achieved memory bandwidth of your final submission, and what fraction of peak A100 bandwidth (2039 GB/s) does it reach? What do you think is preventing you from reaching 100%?


If you prefer to document your optimization in a separate PDF, please also make a note here.


# LLM Prompts

If you used an AI tool to complete any part of this assignment, please paste all prompts you used to produce your final code/responses in the box below and answer the following reflection question.

Prompts Used:
*   adjust the code so that it reiterates the computation 100 times, with 5 warmup iterations before that are not counted
*   can you write the code for part c
*   can you write the code for part (d) (i)?




**Reflection: What parts of the AI generated output required modification or improvement? Describe the feedback you gave the tool to produce your final output or any changes you had to make on your own.**

TODO: ANSWER THE QUESTION HERE (DOUBLE-CLICK TO EDIT)